In [1]:
from pathlib import Path
import pandas as pd

file_path = Path("/workspaces/SOLhycool/simulation/results/20220101_20221231/andasol_pilot_plant_wct100_dc100/sim_results.h5")
df = pd.read_hdf(file_path, key="/results").sort_index()


In [3]:
from typing import get_origin, get_args, Union
from dataclasses import dataclass, field, asdict
import pandas as pd

from solhycool_modeling import OperationPoint


def is_optional_type(tp) -> bool:
    """Return True if the given type annotation is Optional[...]"""
    return get_origin(tp) is Union and type(None) in get_args(tp)


@dataclass
class Scalator:
    """
    Parameters for scaling the system.
    Ratios are in format: target (nominal or max) value / base (nominal or max) value
    
    target_value = target_nominal_value / base_nominal_value x base_value
    
    
    NOTE: Default values correspond to, target=andasol, base=PSA pilot plant
    Andasol values from Table 7 in:
    
    Asfand, F., Palenzuela, P., Roca, L., Caron, A., Lemarié, C.-A., 
    Gillard, J., Turner, P., & Patchigolla, K. (2020). 
    Thermodynamic Performance and Water Consumption of Hybrid Cooling 
    System Configurations for Concentrated Solar Power Plants. 
    Sustainability, 12(11), 4739. https://doi.org/10.3390/su12114739 
    
    (Vapor temperature needs to be estimated since they say 41ºC but thats unfeasible given the ambient conditions)
    """

    thermal_power: float = 90_000 / 200
    water_consumption: float = 276_840.0 / 156 # Andasol: 76.9 kg/s at Tamb=45ºC, HR=60%, Tv=44ºC
    wct_electricity_consumption: float = 500 / 2.8 # Andasol: 4 fans/tower x 3 towers x 41.5 kw/fan = 498 kw
    dc_electricity_consumption: float = 2500 / 5.8 # 
    recirculation_electricity_consumption: float = 250 / 1.57 # Andasol: using value provided by Villena. Pilot plant: recirculation adjusted to same ratio compared to wct as in andasol
    recirculation_flow_rate: float = 11880 / 24 # Andasol: 11880 m3/h, Pilot plant: 24 m3/h
    
    _fields_to_update: list[str] = field(default_factory=lambda: [
        "mv", 
        "qc", "qdc", "qwct", "qwct_s", "qwct_p", 
        "Qc_released", "Qc_absorbed", "Qc_transfered", "Qdc", "Qwct",
        "Ce_wct", "Ce_dc", "Ce_c", 
        "Cw_wct", "Cw_s1", "Cw_s2",
    ])
    _op_pt_optional_fields: list[str] = field(
        default_factory=lambda: [k for k, v in OperationPoint.__annotations__.items() if is_optional_type(v)]
    )
    
    def scale_operation_point(self, op: OperationPoint) -> OperationPoint:
        """Scales an operation point according to the scaling ratios defined in the dataclass.

        Args:
            op (OperationPoint): Operation point to be scaled.

        Returns:
            OperationPoint: Scaled operation point.
        """
        
        # Procedure: create a new OperationPoint instance with the updated values for
        # self._fields_to_update, leave untouched the others which are not Optional fields.
        # The optional fields should be recalculated using the updated values.
        
        
        op_pt_unchanged_fields = {
            k:v
            for k, v in asdict().items() \
            if k not in self._fields_to_update and k not in self._op_pt_optional_fields
        }
        
        updated_fields = {
            "mv": op.mv * self.thermal_power,
            
            "qc": op.qc * self.recirculation_flow_rate,
            "qdc": op.qdc * self.recirculation_flow_rate,
            "qwct": op.qwct * self.recirculation_flow_rate,
            "qwct_s": op.qwct_s * self.recirculation_flow_rate,
            "qwct_p": op.qwct_p * self.recirculation_flow_rate,
            
            "Qc_released": op.Qc_released * self.thermal_power,
            "Qc_absorbed": op.Qc_absorbed * self.thermal_power,
            "Qc_transfered": op.Qc_transfered * self.thermal_power,
            "Qdc": op.Qdc * self.thermal_power,
            "Qwct": op.Qwct * self.thermal_power,
            
            "Ce_wct": op.Ce_wct * self.wct_electricity_consumption,
            "Ce_dc": op.Ce_dc * self.dc_electricity_consumption,
            "Ce_c": op.Ce_c * self.recirculation_electricity_consumption,
            
            "Cw_wct": op.Cw_wct * self.water_consumption,
            "Cw_s1": op.Cw_s1 * self.water_consumption,
            "Cw_s2": op.Cw_s2 * self.water_consumption,
        }
        assert list(updated_fields.keys()) == self._fields_to_update, "Updated fields do not match the fields to update"
        
        return OperationPoint(
            **updated_fields,
            **op_pt_unchanged_fields
        )
        
    def scale_dataframe(self, df_op: pd.DataFrame) -> pd.DataFrame:
        """Scales a dataframe of operation points according to the scaling ratios defined in the dataclass.
        
        NOTE: This method, different from `scale_operation_point`, does not recalculate 
        the optional fields, leaving some inconsistencies in the results.

        Args:
            df_op (pd.DataFrame): Dataframe of operation points to be scaled.

        Returns:
            pd.DataFrame: Scaled dataframe of operation points.
        """
        
        df_scaled = df_op.copy()
        
        for fld in self._fields_to_update:
            if fld in df_scaled.columns:
                if fld == "mv":
                    df_scaled[fld] = df_scaled[fld] * self.thermal_power
                elif fld.startswith("q"): # qc, qdc, qwct, qwct_s, qwct_p
                    df_scaled[fld] = df_scaled[fld] * self.recirculation_flow_rate
                elif fld.startswith("Q"): # Qc_released, Qc_absorbed, Qc_transfered, Qdc, Qwct
                    df_scaled[fld] = df_scaled[fld] * self.thermal_power
                elif fld in ["Ce_wct"]:
                    df_scaled[fld] = df_scaled[fld] * self.wct_electricity_consumption
                elif fld in ["Ce_dc"]:
                    df_scaled[fld] = df_scaled[fld] * self.dc_electricity_consumption
                elif fld in ["Ce_c"]:
                    df_scaled[fld] = df_scaled[fld] * self.recirculation_electricity_consumption
                elif fld.startswith("Cw"): # Cw_wct, Cw_s1, Cw_s2
                    df_scaled[fld] = df_scaled[fld] * self.water_consumption
                else:
                    raise ValueError(f"Field {fld} not recognized for scaling.")
            else:
                print(f"Field {fld} not found in dataframe columns, skipping scaling for this field.")
        
        return df_scaled
    
df_scaled = Scalator().scale_dataframe(df)
df_scaled


,Tamb,HR,mv,Tv,qc,Tc_in,Tc_out,Tcond,Qc_released,Qc_absorbed,...,Je_c,Je_dc,Je_wct,Jw_wct,Jw_s1,Jw_s2,dc_active,dc_mode,wct_active,wct_mode
time,,,,,,,,,,,,,,,,,,,,,
2022-01-01 10:00:00+00:00,8.1,66.0,86798.070,33.00,7425.0,24.272642,31.727358,33.00,58412.777095,58431.790398,...,4.111357e-05,0.001335,0.010371,0.002878,0.002878,0.0,True,0,True,0
2022-01-01 11:00:00+00:00,10.1,53.0,120430.890,38.70,11880.0,28.769034,35.198369,38.70,80592.003861,80609.301406,...,1.488503e-04,0.001504,0.014433,0.004726,0.004726,0.0,True,0,True,0
2022-01-01 12:00:00+00:00,11.7,46.0,94731.966,34.36,10395.0,25.423519,31.227602,34.36,63666.851103,63689.538959,...,1.079032e-04,0.001537,0.017186,0.003863,0.003863,0.0,True,0,True,0
2022-01-01 13:00:00+00:00,12.8,44.0,78260.508,31.52,11880.0,25.208200,29.206694,31.52,52743.779306,50149.281147,...,1.536580e-04,0.001553,0.015380,0.002983,0.002983,0.0,True,0,True,0
2022-01-01 14:00:00+00:00,13.4,42.0,80272.728,31.87,11880.0,25.148067,29.313460,31.87,54081.354803,52242.432967,...,1.547892e-04,0.001564,0.016912,0.003189,0.003189,0.0,True,0,True,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-12-31 17:00:00+00:00,8.4,60.0,89615.178,33.48,8910.0,25.068998,31.411002,33.48,60280.168315,59650.919432,...,1.516423e-05,0.000318,0.002466,0.003009,0.003009,0.0,True,0,True,0
2022-12-31 18:00:00+00:00,7.6,67.0,75644.622,31.07,8910.0,24.323154,29.746846,31.07,51003.284784,51018.944515,...,1.373121e-05,0.000288,0.002233,0.002379,0.002379,0.0,True,0,True,0
2022-12-31 19:00:00+00:00,6.8,71.0,78432.984,31.55,11880.0,25.203147,29.215848,31.55,52858.465244,50327.463223,...,1.194071e-05,0.000121,0.000937,0.002248,0.002248,0.0,True,0,True,0


In [10]:
OperationPoint.__annotations__.values()


dict_values([<class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'float'>, <class 'datetime.datetime'>, typing.Optional[float], typing.Optional[float], typing.Optional[float], typing.Optional[float], typing.Optional[float], typing.Optional[float], typing.Optional[float], typing.Optional[float], typing.Optional[float], typing.Optional[float], typing.Optional[float], typing.Optional[float], typing.Optional[float], 

In [ ]:
OperationPoint.__dataclass_fields__.keys()


dict_keys(['Tamb', 'HR', 'mv', 'Tv', 'qc', 'Tc_in', 'Tc_out', 'Tcond', 'Qc_released', 'Qc_absorbed', 'Qc_transfered', 'Ce_c', 'Rp', 'Rs', 'wdc', 'qdc', 'Tdc_in', 'Tdc_out', 'Ce_dc', 'Qdc', 'wwct', 'qwct', 'qwct_s', 'qwct_p', 'Twct_in', 'Twct_out', 'Ce_wct', 'Cw_wct', 'Qwct', 'Tcc_out', 'Cw_s1', 'Cw_s2', 'Pw', 'Pw_s1', 'Pw_s2', 'Pe', 'Vavail', 'deltaV', 'time', 'qdc_only', 'Tcc_in', 'Ce_cc', 'Cw_cc', 'Qcc', 'Ce', 'Cw', 'qcc', 'Vavail_s1', 'Je', 'Jw', 'J', 'Je_c', 'Je_dc', 'Je_wct', 'Jw_wct', 'Jw_s1', 'Jw_s2', 'dc_active', 'dc_mode', 'wct_active', 'wct_mode'])

In [19]:
# Test

# Leer datos de entorno andasol downscaled
# Para cada hora donde Tv > 0, estimar Twb
# Calcular diferencia de temperatura entre Tv y Twb, contar valores con diferencia menor de 5ºC

import pandas as pd
import psychrolib
import numpy as np


psychrolib.SetUnitSystem(psychrolib.SI)
get_Twb = np.vectorize(psychrolib.GetTWetBulbFromRelHum)

df_env = pd.read_hdf("/workspaces/SOLhycool/data/datasets/environment_data_psa_andasol_20220101_20241231.h5")
df_env = df_env[df_env["Tv_C"] > 0].copy()
df_env["Twb"] = get_Twb(df_env["Tamb_C"].values, df_env["HR_pct"]/100,  np.full(len(df_env), 101325))

df_env["Tv_Twb_diff"] = df_env["Tv_C"] - df_env["Twb"]
df_env_close = df_env[np.abs(df_env["Tv_Twb_diff"]) < 5]

print(f"Number of hours with Tv > 0: {len(df_env)}")


Number of hours with Tv > 0: 10845


In [ ]:
df_env["Tv_Twb_diff"].argmin()


np.int64(1655)

In [26]:
df_env.iloc[df_env["Tv_Twb_diff"].argmax()]



G_Gh                               11.000000
G_Dh                               11.000000
G_Gk                                8.000000
G_Bn                                0.000000
Tamb_C                              4.900000
HR_pct                             56.000000
 hs                                 2.500000
 Az                                71.200000
precip_mm                           0.000000
 Td                                -3.100000
Ce_pvpc_eur_MWh                   409.200000
Ce_spot_market_price_eur_MWh      248.710000
Ce_pvpc_eur_kWh                     0.409200
Ce_spot_market_price_eur_kWh        0.248710
0                                        NaN
Q_kW                              203.829964
Tv_C                               41.510000
mv_kgh                            305.441160
water_price_eur_m3                  0.029054
water_price_alternative_eur_m3     19.896800
water_price_eur_l                   0.000029
water_price_alternative_eur_l       0.019897
Vavail_m3 